# Feature Engineering & ML Preparation
In this notebook, i take the finalized SQL dataset and perform the last transformations before feeding the data into a Machine Learning model. This includes Temporal Engineering, Target Encoding, and dealing with Data Leakage.

In [4]:
import pandas as pd
import numpy as np
import os

### Load the Merged Data
I recreate the merged dataset from Notebook 2.

In [5]:
data_dir = '../data/sql_outputs/'
orders_df = pd.read_csv(os.path.join(data_dir, 'order_features_with_returns.csv'))
customers_df = pd.read_csv(os.path.join(data_dir, 'customer_features.csv'))
products_df = pd.read_csv(os.path.join(data_dir, 'product_features.csv'))

df_merged = pd.merge(orders_df, customers_df, on='customer_unique_id', how='left', suffixes=('', '_cust'))
category_risk_df = products_df.groupby('product_category_name').agg({
    'high_complaint_product': 'mean',
    'high_dissatisfaction_rate': 'mean',
    'avg_review_score': 'mean',
    'low_rating_percentage': 'mean'
}).reset_index().rename(columns={
    'high_complaint_product': 'category_complaint_rate',
    'high_dissatisfaction_rate': 'category_dissatisfaction_rate',
    'avg_review_score': 'category_avg_rating',
    'low_rating_percentage': 'category_low_rating_pct'
})
df_final = pd.merge(df_merged, category_risk_df, left_on='product_category', right_on='product_category_name', how='left')
df_final.drop(columns=['product_category_name'], inplace=True, errors='ignore')
print(f"Initial Shape: {df_final.shape}")

Initial Shape: (96999, 45)


### Temporal Feature Engineering
Raw timestamps (like `2017-10-02 10:56:33`) are useless to an ML algorithm. We must extract the month and day of the week to capture seasonal trends.

In [6]:
df_final['order_purchase_timestamp'] = pd.to_datetime(df_final['order_purchase_timestamp'])
df_final['purchase_month'] = df_final['order_purchase_timestamp'].dt.month
df_final['purchase_day_of_week'] = df_final['order_purchase_timestamp'].dt.dayofweek

print("Extracted Temporal Features:")
print(df_final[['order_purchase_timestamp', 'purchase_month', 'purchase_day_of_week']].head())

Extracted Temporal Features:
  order_purchase_timestamp  purchase_month  purchase_day_of_week
0      2017-10-02 10:56:33              10                     0
1      2018-07-24 20:41:37               7                     1
2      2018-08-08 08:38:49               8                     2
3      2017-11-18 19:28:06              11                     5
4      2018-02-13 21:18:39               2                     1


### Target Encoding
We have high-cardinality categorical variables like `customer_city` and `customer_state`. One-hot encoding them would create thousands of useless columns. Instead, we use Target Encoding: replacing the category string with its historical return rate.

In [7]:
# Target encode State
state_target_mean = df_final.groupby('customer_state')['is_returned'].mean()
df_final['state_return_rate'] = df_final['customer_state'].map(state_target_mean)

# Target encode City
city_target_mean = df_final.groupby('customer_city')['is_returned'].mean()
df_final['city_return_rate'] = df_final['customer_city'].map(city_target_mean)

print("Target Encoded Geography:")
print(df_final[['customer_state', 'state_return_rate']].drop_duplicates().head())

Target Encoded Geography:
  customer_state  state_return_rate
0             SP           0.159464
1             BA           0.227925
2             GO           0.183891
3             RN           0.176471
5             PR           0.162282


### Handling Missing Values
ML algorithms crash if they encounter `NaN` values. We will impute numerical categories with the median, and fill edge cases with 0.

In [8]:
cols_to_fill_median = [
    'category_complaint_rate', 
    'category_dissatisfaction_rate', 
    'category_avg_rating', 
    'category_low_rating_pct'
]

for col in cols_to_fill_median:
    df_final[col] = df_final[col].fillna(df_final[col].median())

# Fill any remaining nulls with 0
df_final = df_final.fillna(0)
print(f"Remaining Missing Values: {df_final.isnull().sum().sum()}")

Remaining Missing Values: 0


### Dropping Data Leakage & Identifiers
If we feed `return_probability_score` into the model, the model will cheat and achieve 100% fake accuracy (Data Leakage). We must also drop raw ID strings since they cause overfitting.

In [10]:
cols_to_drop = [
    'order_id', 'customer_unique_id', 'order_purchase_timestamp', 
    'customer_city', 'customer_state', 'product_category',
    'return_probability_score' 
]

df_ml = df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns])

print("Columns dropped to prevent Data Leakage & Overfitting.")
print(f"Final ML Shape: {df_ml.shape}")

Columns dropped to prevent Data Leakage & Overfitting.
Final ML Shape: (96999, 42)


### 6. Save the Final Dataset

In [11]:
os.makedirs('../data/final', exist_ok=True)
df_ml.to_csv('../data/final/ml_ready_dataset.csv', index=False)
print("SUCCESS! Saved ml_ready_dataset.csv to data/final/")

SUCCESS! Saved ml_ready_dataset.csv to data/final/
